In [1]:
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    array_join, array_remove, col, current_timestamp, datediff,
    lit, to_date, when
)
from pyspark.sql.types import BooleanType, IntegerType
from pyspark.sql import Window
import pyspark.sql.functions as F

SRC_TRANSACTIONS = "silver_transactions"
SRC_ACCOUNTS     = "silver_accounts"
DIM_ACCOUNT      = "dim_account"          # Gold, Lakehouse
DIM_DATE         = "dim_date"             # Gold, Lakehouse
FACT_TXN         = "fact_transactions"    # Gold, Lakehouse
FACT_SUSP        = "fact_suspicious_transactions"  # Gold, Lakehouse
WATERMARK_TABLE  = "_pipeline_watermarks"
LAYER_KEY_TXN    = "gold_fact_transactions"
LAYER_KEY_SUSP   = "gold_fact_suspicious"

# Fraud rule thresholds
RULE1_AMOUNT_THRESHOLD    = 5000
RULE2_VELOCITY_WINDOW_SEC = 3600
RULE2_VELOCITY_MAX_COUNT  = 3
RULE3_MISMATCH_AMOUNT_MIN = 1000
RULE4_ACCOUNT_AGE_DAYS    = 548
RULE4_MIN_AMOUNT          = 2000
RULE5_MIN_AMOUNT          = 500

print("Config loaded.")

StatementMeta(, 65eebae1-e962-476a-b1f8-f50f5ff317b6, 3, Finished, Available, Finished, False)

Config loaded.


In [2]:
last_watermark = (
    spark.table(WATERMARK_TABLE)
         .filter(f"layer_name = '{LAYER_KEY_TXN}'")
         .select("watermark_ts")
         .collect()[0]["watermark_ts"]
)
print(f"Gold facts watermark: {last_watermark}")

StatementMeta(, 65eebae1-e962-476a-b1f8-f50f5ff317b6, 4, Finished, Available, Finished, False)

Gold facts watermark: 2026-06-19 19:33:14.934004


In [3]:
df_new_silver = (
    spark.table(SRC_TRANSACTIONS)
         .filter(col("silver_updated_at") > last_watermark)
)
new_count = df_new_silver.count()
print(f"New Silver rows: {new_count}")

if new_count == 0:
    print("No new rows. Exiting.")
    spark.stop()
    notebookutils.notebook.exit("NO_NEW_DATA")

StatementMeta(, 65eebae1-e962-476a-b1f8-f50f5ff317b6, 5, Finished, Available, Finished, False)

New Silver rows: 0
No new rows. Exiting.
ExitValue: NO_NEW_DATA

In [ ]:
# CRITICAL: join on is_current = True to capture the CURRENT account version's key.
# Historical transactions already have their key stored in fact_transactions.
df_acct_keys = (
    spark.table(DIM_ACCOUNT)
         .filter(col("is_current") == True)
         .select("account_id", "account_key")
)
df_date_keys = spark.table(DIM_DATE).select("date_key", "full_date")

df_keyed = (
    df_new_silver
    .withColumn("txn_date", to_date("transaction_timestamp"))
    .join(df_acct_keys, on="account_id", how="left")
    .join(df_date_keys, col("txn_date") == col("full_date"), how="left")
    .drop("txn_date", "full_date")
)

missing_keys = df_keyed.filter(col("account_key").isNull()).count()
print(f"Keyed rows: {df_keyed.count()}  |  Missing account_keys: {missing_keys}")

StatementMeta(, 65eebae1-e962-476a-b1f8-f50f5ff317b6, -1, Cancelled, , Cancelled, True)

In [ ]:
df_new_fact_txn = df_keyed.select(
    "transaction_id", "account_key", "date_key",
    "amount", "merchant", "city", "source_system",
    "is_city_mismatch", "risk_category", "account_status",
    "transaction_timestamp",
)

StatementMeta(, 65eebae1-e962-476a-b1f8-f50f5ff317b6, -1, Cancelled, , Cancelled, True)

In [ ]:
df_acct_extra = spark.table(SRC_ACCOUNTS).select(
    "account_id", "account_open_date", "kyc_status"
)

df_active = (
    df_keyed.filter(col("account_status") == "ACTIVE")
            .drop("kyc_status")
            .join(df_acct_extra, on="account_id", how="left")
)

# Rule 1: HIGH_VALUE
df_r = df_active.withColumn("flag_high_value",
    (col("amount") > RULE1_AMOUNT_THRESHOLD).cast(BooleanType()))

# Rule 2: VELOCITY (Window Function)
window_v = (Window.partitionBy("account_id")
                  .orderBy(F.unix_timestamp("transaction_timestamp"))
                  .rangeBetween(-RULE2_VELOCITY_WINDOW_SEC, 0))
df_r = (df_r
    .withColumn("txn_count_in_window",
        F.count("transaction_id").over(window_v).cast(IntegerType()))
    .withColumn("flag_velocity",
        (col("txn_count_in_window") > RULE2_VELOCITY_MAX_COUNT).cast(BooleanType())))

# Rule 3: GEO_MISMATCH
df_r = df_r.withColumn("flag_geo_mismatch",
    ((col("is_city_mismatch") == True) &
     (col("amount") > RULE3_MISMATCH_AMOUNT_MIN)).cast(BooleanType()))

# Rule 4: NEW_ACCOUNT_LARGE_TXN
df_r = (df_r
    .withColumn("account_age_days",
        datediff(to_date("transaction_timestamp"), col("account_open_date")).cast(IntegerType()))
    .withColumn("flag_new_account",
        ((col("account_age_days") < RULE4_ACCOUNT_AGE_DAYS) &
         (col("amount") > RULE4_MIN_AMOUNT)).cast(BooleanType())))

# Rule 5: KYC_NON_COMPLIANT
df_r = df_r.withColumn("flag_kyc_non_compliant",
    (col("kyc_status").isin("EXPIRED", "PENDING") &
     (col("amount") > RULE5_MIN_AMOUNT)).cast(BooleanType()))

# Compose rules_triggered string and rule_count
df_r = df_r.withColumn("rules_triggered",
    array_join(array_remove(F.array(
        when(col("flag_high_value"),       lit("HIGH_VALUE")),
        when(col("flag_velocity"),          lit("VELOCITY")),
        when(col("flag_geo_mismatch"),      lit("GEO_MISMATCH")),
        when(col("flag_new_account"),       lit("NEW_ACCOUNT_LARGE_TXN")),
        when(col("flag_kyc_non_compliant"), lit("KYC_NON_COMPLIANT")),
    ), None), "|")
).withColumn("rule_count",
    col("flag_high_value").cast(IntegerType()) +
    col("flag_velocity").cast(IntegerType()) +
    col("flag_geo_mismatch").cast(IntegerType()) +
    col("flag_new_account").cast(IntegerType()) +
    col("flag_kyc_non_compliant").cast(IntegerType())
)

df_new_suspicious = (
    df_r.filter(col("rule_count") > 0)
    .withColumn("suspicious_id", (F.monotonically_increasing_id() + 1).cast("long"))
    .withColumn("flagged_at", current_timestamp())
    .select("suspicious_id","transaction_id","account_key","date_key","amount",
            "merchant","city","risk_category","rules_triggered","rule_count",
            "flag_high_value","flag_velocity","flag_geo_mismatch",
            "flag_new_account","flag_kyc_non_compliant",
            "txn_count_in_window","account_open_date","kyc_status",
            "transaction_timestamp","flagged_at")
)

print(f"New fact_transactions rows     : {df_new_fact_txn.count()}")
print(f"New fact_suspicious rows       : {df_new_suspicious.count()}")

StatementMeta(, 65eebae1-e962-476a-b1f8-f50f5ff317b6, -1, Cancelled, , Cancelled, True)

In [ ]:
def delta_merge(df_new, table_name, merge_key):
    if spark.catalog.tableExists(table_name):
        dt = DeltaTable.forName(spark, table_name)
        (dt.alias("t").merge(df_new.alias("s"), f"t.{merge_key} = s.{merge_key}")
           .whenMatchedUpdateAll()
           .whenNotMatchedInsertAll()
           .execute())
        print(f"MERGE complete: {table_name}")
    else:
        df_new.write.mode("overwrite").format("delta").saveAsTable(table_name)
        print(f"First run — wrote {table_name}")

delta_merge(df_new_fact_txn,    FACT_TXN,  "transaction_id")
delta_merge(df_new_suspicious,  FACT_SUSP, "transaction_id")

StatementMeta(, 65eebae1-e962-476a-b1f8-f50f5ff317b6, -1, Cancelled, , Cancelled, True)

In [ ]:
max_silver_ts = df_new_silver.agg(F.max("silver_updated_at")).collect()[0][0]

for layer, count in [(LAYER_KEY_TXN, df_new_fact_txn.count()),
                     (LAYER_KEY_SUSP, df_new_suspicious.count())]:
    spark.sql(f"""
        UPDATE {WATERMARK_TABLE}
        SET watermark_ts = '{max_silver_ts}', rows_last_run = {count},
            last_run_status = 'SUCCESS', last_run_at = current_timestamp()
        WHERE layer_name = '{layer}'
    """)

print(f"Watermarks updated to: {max_silver_ts}")

spark.stop()
print("Gold facts incremental complete.")

StatementMeta(, 65eebae1-e962-476a-b1f8-f50f5ff317b6, -1, Cancelled, , Cancelled, True)